# Impacting oscillator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bernardorivas/hybrid_boxmap/blob/main/notebooks/impacting_oscillator.ipynb)

The system is given by

$$\dot x=v,\qquad \dot v=x-x^3+\varepsilon(\beta-x^2)\,v,\qquad x\leq w,$$

with guard and reset

$$G=\{(w,v): v\geq 0\},\qquad r(w,v)=(w,-cv),$$

where $\varepsilon=1$, $\beta=0.76$, $w=0.8$, and $c=0.7$.

Stored figures are redrawn below. [Recompute from scratch](impacting_oscillator_from_scratch.ipynb).


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError("Python 3.12 or newer is required; select a current Colab runtime.")

if "google.colab" in sys.modules:
    repo = Path("/content/hybrid_boxmap")
    if not repo.exists():
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/bernardorivas/hybrid_boxmap.git", str(repo)], check=True)
    # Installation also runs when the checkout already exists.
    if shutil.which("dot") is None:
        subprocess.run(["apt-get", "-qq", "update"], check=True)
        subprocess.run(["apt-get", "-qq", "install", "-y", "graphviz"], check=True)
    subprocess.run([
        sys.executable, "-m", "pip", "install", "--quiet", "-e", str(repo),
        "cmgdb==1.3.3+fork.6", "--find-links",
        "https://github.com/bernardorivas/CMGDB/releases/expanded_assets/v1.3.3%2Bfork.6",
    ], check=True)
else:
    here = Path.cwd().resolve()
    candidates = [here, *here.parents, here / "code"]
    repo = next((p for p in candidates if (p / "demo/run_paper_examples.py").is_file()), None)
    if repo is None:
        raise RuntimeError("Open this notebook from the hybrid_boxmap checkout.")

os.chdir(repo)
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))
if shutil.which("dot") is None:
    raise RuntimeError("Install Graphviz (the dot executable) before plotting Morse graphs.")
import CMGDB
from hybrid_dynamics.src.carrier_kernel import cmgdb_provenance
if not hasattr(CMGDB, "ComputeCarrierChainMap"):
    raise RuntimeError("Install CMGDB 1.3.3+fork.6 as described in the README.")
print("Repository:", repo)
print("CMGDB:", cmgdb_provenance("native"))


## Phase portrait

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from hybrid_dynamics.examples.paper_examples import paper_problem

problem = paper_problem("impact-vdp-duffing-beta076")

fig, ax = plt.subplots(figsize=(5, 4))
for state in [(-1.5, 0.0), (0.3, 0.0), (-0.5, 1.5), (0.7, 0.05), (0.75, -0.1), (-1.9, 1.9), (0.0, 0.05)]:
    trajectory = problem.system.simulate(np.array(state, float), (0.0, 40.0), max_jumps=200, max_step=0.02)
    color = None
    for segment in trajectory.segments:
        line, = ax.plot(*np.asarray(segment.state_values).T, lw=0.8, color=color)
        color = line.get_color()
ax.plot([0.8, 0.8], [0, 1.95], color="k", lw=2.5)  # guard
ax.set(xlim=(-1.95, 0.82), ylim=(-2.35, 1.95), xlabel="x", ylabel="v")
plt.show()

## Morse sets

Stored computation: $\tau=0.5$, level $n=7$, offset $K=4$, $2048^2$ base cells, and 512 phase cells on $R=[-1.95,0.8]\times[-2.35,1.95]$.

Base cells per axis are $2^{n+K}$; phase cells are $2^{n+2}$. Gap refinement depth is 14.


In [ ]:
from datetime import datetime, timezone
from uuid import uuid4
from IPython.display import Image, display

output_dir = Path("output") / ("notebook-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid4().hex[:8])
output_dir.mkdir(parents=True, exist_ok=False)
print("Output:", output_dir.resolve())


In [ ]:
from demo.replot_paper import replot

record = Path("figures/paper/paper-impact-vdp-duffing-beta076-tau050-level7-base2048-corners-gap-refined.json")
replot(record, output_dir=output_dir, update_json=False)
stem = output_dir / record.stem
for panel in ['nontrivial-base', 'nontrivial-zoom-A', 'nontrivial-zoom-B']:
    path = Path(f"{stem}-{panel}.png")
    if path.exists():
        display(Image(str(path), width=520))


## Conley–Morse graph

Labels give index dimensions in degrees 0, 1, 2. Morse sets with trivial Conley index are gray.

In [ ]:
for panel in ["nontrivial-graph", "graph"]:
    display(Image(f"{stem}-{panel}.png", width=520))


## Attractor lattice

The down-sets of the order restricted to the Morse sets with nontrivial index.

In [ ]:
display(Image(f"{stem}-attractor-lattice.png", width=520))
